# Plot regional mortality due to exposure to PM<sub>2.5</sub> as a box plot

Using CESM2, SSP2-4.5 ensemble member 1 as an example

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from utils.utils import get_scenario_config

In [ ]:
def process_mortality_per_100k(model, scenario, config, ens_num):
    # === Path config ===
    MASK_DIR = "/glade/work/awells/air_quality/BMR/masks/region/"
    POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
    MORT_DIR = f"/glade/work/awells/workflow/{model}/mortality/pm25/"

    # === Load data ===
    mask_file = "GBD_Region_Masks_0.10.nc"
    mask_path = os.path.join(MASK_DIR, mask_file)
    mask = xr.open_dataarray(mask_path)

    # Regional population
    pop_file = "ssp2_region_level_2000-2100.nc"
    pop_path = os.path.join(POP_DIR, pop_file)
    population = xr.open_dataarray(pop_path)
    pop = population.reindex_like(mask, method="nearest", tolerance=1e-9)

    years = config["years"]
    dates = f"{years.start}-{years.stop}"

    # Regional mortality
    file = f"Regional_mortality_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    file_path = os.path.join(MORT_DIR, file)
    da = xr.open_dataarray(file_path)

    # Calculate end of scenario mean (10 years)
    end_idx = da.sizes["year"]
    # Select the last 10 years
    da_last = da.isel(year=slice(end_idx - 10, end_idx))

    # Get the first and last time values
    start_time = da_last.year[0].item()
    end_time = da_last.year[-1].item()

    # Calculate temporal mean
    da_mean = da_last.mean("year")

    # Calculate population mean
    years = slice(start_time, end_time)
    av_pop = pop.sel(year=years).mean("year")

    # Calculate mortality per 100,000
    da_per_100k = (da_mean/av_pop)*100000
    return da_per_100k, start_time, end_time

In [ ]:
def plot_regional_boxplot(da, model, scenario, ens_num, start_year, end_year, SAVE_DIR):
    # Sort regions by mean value
    region_means = da.mean(dim="samples")
    sorted_regions = region_means.sortby(region_means).region.values

    data = [da.sel(region=r).values for r in sorted_regions]

    fcolor = "#FEA571"

    # --- Plot ---
    plt.figure(figsize=(8, 10))
    plt.boxplot(data, vert=False, patch_artist=True, sym="+",
                widths=0.6,
                boxprops=dict(facecolor=fcolor, color="gray", lw=0.5),
                medianprops=dict(color="white", lw=1),
                whiskerprops=dict(color="gray"),
                capprops=dict(color="gray"),
                flierprops=dict(color="gray", alpha=0.4))

    plt.yticks(np.arange(1, len(sorted_regions) + 1), sorted_regions)
    plt.xlabel("Total PM$_{2.5}$ Mortality per 100,000 (year$^{-1}$)")
    plt.title(f"{model} {scenario} ensemble {ens_num}\n {first_year}-{last_year}")
    plt.grid(axis="y", linestyle=':', alpha=0.5)
    plt.grid(axis='x', linestyle='--', alpha=0.8)

    # Use ScalarFormatter to show tick labels as real numbers
    formatter = ScalarFormatter()
    formatter.set_scientific(False)
    plt.gca().xaxis.set_major_formatter(formatter)

    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.tick_params(axis="x", length=0)
    ax.tick_params(axis="y", length=0)

    plt.tight_layout()

    out_file = f"Mortality_per_100k_pm25_region_boxplot_{model}_{scenario}_{first_year}-{last_year}.png"
    out_path = os.path.join(SAVE_DIR, out_file)
    plt.savefig(out_path, dpi=300)
    return

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
SAVE_DIR = "/glade/u/home/awells/air_quality_project/plotting/example_workflow/"

model = "CESM2"
scenario = "SSP245"
ensemble_number = 1

n_samples = 300

config = get_scenario_config(model, scenario)

da_100k, first_year, last_year = process_mortality_per_100k(model, scenario, config, ensemble_number)

# Plotting
plt.rcParams.update({'font.size': 16})
plot_regional_boxplot(da_100k, model, scenario, ensemble_number, first_year, last_year, SAVE_DIR)